In [ ]:
import json
from dotenv import load_dotenv

load_dotenv()

import importlib

import agents

importlib.reload(agents)

from agents.templates.play_zero_agent import PlayZeroAgent
from agents.structs import FrameData, GameState

import textwrap

def print_wrapped_text(text: str, width: int = 80):
    """
    Print the given text with word-wrapped lines for better readability in the terminal.

    Args:
        text (str): The input text to be printed.
        width (int): The maximum line width before wrapping. Default is 80.
    """
    wrapper = textwrap.TextWrapper(width=width)
    paragraphs = text.strip().split("\n\n")

    for paragraph in paragraphs:
        wrapped = wrapper.fill(paragraph)
        print(wrapped + "\n")

#  Agent.__init__() missing 5 required positional arguments: 'card_id', 'game_id', 'agent_name', 'ROOT_URL', and 'record'
play_zero_agent: PlayZeroAgent = PlayZeroAgent(
    card_id="play_zero_agent",
    game_id="play_zero_agent",
    agent_name="PlayZeroAgent",
    ROOT_URL="http://localhost:8000",
    record=False,
)

video_path = "/workspaces/ARC-AGI-3-Agents/recordings/game_analysis_ls20-f340c8e5138e_track1.mp4"
scorecard_file_path = "/workspaces/ARC-AGI-3-Agents/recordings/ls20-f340c8e5138e.playzeroagent.gemini-2.5-flash.with-observe.gemini-2.5-flash.8aefc1ea-8e65-41ec-9272-a8faff24ddb1.recording.jsonl"
scorecard_file_path_track2 = "recordings/ls20-f340c8e5138e.playzeroagent.gemini-2.5-flash.with-observe.gemini-2.5-flash.fa1f97de-edb6-47c7-98bb-eff9f689d487.recording.jsonl"
with open(scorecard_file_path, "r") as file:
    grid_jsons = [json.loads(line) for line in file]
frames = [FrameData(**frame_json["data"]) for frame_json in grid_jsons]
frame_start = frames[0]
frame_2 = frames[1]
frame_3 = frames[2]
frame_end = frames[-3]
current_frame = frame_end

def write_eval_log(evaluation_result):
    with open("eval.log", "a") as eval_log_file:
        print_wrapped_text(f"Evaluation Result:\n\n {evaluation_result}\n")
        eval_log_file.write(f"Evaluation Result:\n\n {evaluation_result}\n")


/workspaces/ARC-AGI-3-Agents/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
logical_analysis_actions_summary = play_zero_agent.generate_logical_analysis_summary(frames[:55])
print(f"Logical Analysis Actions Summary:\n\n {logical_analysis_actions_summary}")

Logical Analysis Actions Summary:

 Out of all user inputs recorded during gameplay:

- **W** was used 15 times, with 4 inputs having no effect on gameplay.
- **A** was used 7 times, and all had an effect on the game.
- **S** was used 5 times, with 2 inputs having no effect on gameplay.
- **D** was used 10 times, with 3 inputs having no effect on gameplay.
- **CLICK** was used 8 times, and all had no effect on the game.


In [7]:
# Eval prompt for multiple_hypothesis_text

EVAL_PROMPT = """Give score and reason of whether the multiple hypothesis can be used to generate the expected goal

Expected Goal: <expected_goal>{expected_goal}</expected_goal>

Multiple Hypothesis Text: <multiple_hypothesis_text>{multiple_hypothesis_text}</multiple_hypothesis_text>

Example output json:
```json
{{
    "reason": "<max of 100 words>",
    "score": "<float score from 0.0 to 1.0>"
}}
```
"""

GOAL_RELEVANCE_PROMPT = """Give score and reason of whether the generated goal can be used to achieve the expected goal

Expected Goal: <expected_goal>{expected_goal}</expected_goal>

Generated Goal: <generated_goal>{generated_goal}</generated_goal>

Example output json:
```json
{{
    "reason": "<max of 100 words>",
    "score": "<float score from 0.0 to 1.0>"
}}
```
"""


def evaluate_multiple_hypothesis_text(multiple_hypothesis_text: str, expected_goal: str):
    prompt = EVAL_PROMPT.format(
        expected_goal=expected_goal,
        multiple_hypothesis_text=multiple_hypothesis_text
    )
    response = play_zero_agent.client.chat.completions.create(
        model="gemini-2.5-flash",
        messages = [
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    json_text = response.choices[0].message.content.strip()
    json_text = play_zero_agent.extract_first_json_block(json_text)
    json_data = json.loads(json_text)
    return json_data

def eval_goal_relevance(generated_goal: str, expected_goal: str):
    prompt = GOAL_RELEVANCE_PROMPT.format(
        expected_goal=expected_goal,
        generated_goal=generated_goal
    )
    response = play_zero_agent.client.chat.completions.create(
        model="gemini-2.5-flash",
        messages = [
            {
                "role": "user",
                "content": prompt,
            }
        ]
    )
    json_text = response.choices[0].message.content.strip()
    json_text = play_zero_agent.extract_first_json_block(json_text)
    json_data = json.loads(json_text)
    return json_data


In [4]:

multiple_hypothesis_text = play_zero_agent.generate_multiple_random_hypothesis_from_video(
    video_path,
    logical_analysis_actions_summary=logical_analysis_actions_summary,
)
expected_goal_ls20_1 = "You need to move the \"Orange-Capped Blue Block (6x7)\" to the target \"8x7Grid_BlackHead_BlueEye_WhiteSnout\""
evaluation_result = evaluate_multiple_hypothesis_text(
    multiple_hypothesis_text=multiple_hypothesis_text,
    expected_goal=expected_goal_ls20_1
)
write_eval_log(evaluation_result)

Evaluation Result:

 {'reason': "The multiple hypothesis text accurately identifies the movable
object as the 'blue rectangular block with an orange bar on top,' which clearly
corresponds to the 'Orange-Capped Blue Block.' It also precisely describes the
target as the 'black square (target area)' with a 'small blue dot,' which is a
descriptive match for '8x7Grid_BlackHead_BlueEye_WhiteSnout.' The goal of
'moving' this block to the target is fully explained through the 'Pushing' and
'Achieving the Goal (Alignment)' hypotheses.", 'score': 0.95}



In [8]:
goal = play_zero_agent.generate_top_hypothesis(
    multiple_hypothesis_text=multiple_hypothesis_text,
    logical_analysis_actions_summary=logical_analysis_actions_summary,
)
goal_relevance_result = eval_goal_relevance(
    generated_goal=goal,
    expected_goal=expected_goal_ls20_1
)
write_eval_log(goal_relevance_result)
print_wrapped_text(f"Goal:\n\n {goal}"
)

Evaluation Result:

 {'reason': 'The generated goal accurately describes the movable object ("blue
rectangular block with an orange bar on top" vs. "Orange-Capped Blue Block") and
the action (moving/pushing). However, the target description significantly
diverges. The expected goal provides a specific, named target
("8x7Grid_BlackHead_BlueEye_WhiteSnout"), while the generated goal describes
generic geometric components ("black square (target area)" with a "small blue
dot"). This difference in target identification makes it challenging for a user
to directly confirm alignment with the expected goal, even if the underlying
mechanics are correct.', 'score': 0.6}

Goal:

 Goal: The **final goal** is for the **small blue square (player character)**, a
small blue square, to maneuver the **blue rectangular block with an orange bar
on top**, a rectangular blue block with an orange bar on its top, by pushing it.
The objective is to achieve precise alignment of the **orange bar** of the
**blue r

In [12]:
# print_wrapped_text(f"Multiple Hypothesis Text:\n\n {multiple_hypothesis_text}")
print_wrapped_text(f"Logical Analysis Actions Summary:\n\n {logical_analysis_actions_summary}")
# print_wrapped_text(f"Goal:\n\n {goal}")

Logical Analysis Actions Summary:

 Out of all user inputs recorded during gameplay:

- **W** was used 15 times, with 4 inputs having no effect on gameplay. - **A**
was used 7 times, and all had an effect on the game. - **S** was used 5 times,
with 2 inputs having no effect on gameplay. - **D** was used 10 times, with 3
inputs having no effect on gameplay. - **CLICK** was used 8 times, and all had
no effect on the game.

